## Part 4: Background Subtraction

Diffuse scattering experiments are different from Bragg data collection in that background must be measured carefully and subtracted. For insulin, background images were collected using 1 second exposures every 1 degree of rotation (compared with 0.1 second / 0.1 degree for the crystal). The background dataset was previously imported using _DIALS_ in Part 1 to produce the file `background.expt`. First, import the dataset using `mdx2.import_data`

In [1]:
!mdx2.import_data background.expt --outfile bkg_data.nxs

14:43:09 INFO    | Starting mdx2.import_data at 2025-12-12 14:43:09
14:43:09 INFO    | Parameters(expt='background.expt', chunks=None, outfile='bkg_data.nxs', nproc=1, datastore='datastore')
14:43:09 INFO    | Loading experiment metadata...


14:43:09 INFO    | Image data shape (phi, iy, ix): (50, 2527, 2463)
14:43:09 INFO    | Creating virtual dataset structure...


14:43:09 INFO    | Writing 10 image batches (requested n_jobs: 1)...
14:43:09 INFO    | Using backend: SequentialBackend, n_jobs: 1


[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    6.2s


[Parallel(n_jobs=1)]: Done   4 tasks      | elapsed:   16.0s


[Parallel(n_jobs=1)]: Done   7 tasks      | elapsed:   24.5s


[Parallel(n_jobs=1)]: Done  10 out of  10 | elapsed:   33.0s finished
14:43:42 INFO    | Image data writing completed
14:43:42 INFO    | Image data import completed successfully
14:43:42 SUCCESS | mdx2.import_data completed in 33.25 seconds


Inspect the images in `bkg_data.nxs` using _NeXpy_. The background pattern includes scattering from air, diffuse rings from the capillary surrounding the crystal, and a shadow of the pin. 

Because the background features vary gradually across the detector and with rotation angle, the noise can be reduced by smoothing. In _mdx2_, a downsampling (binning) method is used. 

In [2]:
!mdx2.bin_image_series bkg_data.nxs 10 20 20 --valid_range 0 200 --outfile bkg_data_binned.nxs

14:43:43 INFO    | Starting mdx2.bin_image_series at 2025-12-12 14:43:43
14:43:43 INFO    | Parameters(data='bkg_data.nxs', bins=(10, 20, 20), mask=None, valid_range=(0, 200), outfile='bkg_data_binned.nxs', nproc=1)
14:43:43 INFO    | Loading image series...
14:43:44 INFO    | Original image shape: (50, 2527, 2463)
14:43:44 INFO    | Binning images with bin size: (10, 20, 20)
14:43:44 INFO    | Binning 5 image slabs (requested n_jobs: 1)...
14:43:44 INFO    | Using backend: SequentialBackend, n_jobs: 1


[Parallel(n_jobs=1)]: Done   1 tasks      | elapsed:    1.8s


[Parallel(n_jobs=1)]: Done   4 tasks      | elapsed:    7.1s


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:    8.8s finished
14:43:52 INFO    | Binning computation completed
14:43:52 INFO    | Binned image shape: (5, 127, 124)
14:43:52 INFO    | Saving binned data to bkg_data_binned.nxs...


14:43:52 INFO    | Binning completed successfully
14:43:52 SUCCESS | mdx2.bin_image_series completed in 8.91 seconds


The function has three mandatory arguments after the input file name to specify the bin size (here, 10 degrees by 20 pixels by 20 pixels). In addition, the optional argument `valid_range` is used to mask any pixel with counts outside the given interval. Here, a maximum of 200 counts was chosen to be 10 times the nominal background level of ~20 counts per pixel. The threshold is used to reject broken pixels and stray diffraction if present (e.g. from tiny salt crystals), and is similar to `count_threshold` in `mdx2.find_peaks` described in Part 3.

The resulting binned image stack is saved as `bkg_data_binned.nxs` and can be viewed using _NeXpy_. The binned images will be used after integration when applying geometric corrections in Part 5.